# SLM vs flagship LLMs on financial report QA (TAT-QA)

Everything lives in the GitHub repo. This notebook only **pulls the repo, loads your Kaggle Secrets, and runs the pipeline**.
To change anything (settings, models, prices, which steps run), edit `kaggle/config.toml` on GitHub and re-run. Never edit these cells.

**One-time setup**
- Settings: **Accelerator = GPU T4 x2**, **Internet = On**
- Add-ons -> Secrets (tick each one for this notebook): `GITHUB_TOKEN` (required), `HF_TOKEN`, `LLM_API_KEY`, `ANTHROPIC_API_KEY` (optional)

**How to run**
- Quick check (interactive, ~15 min): run cell 1, then cell 2 with `STAGES = "setup,data,dryrun"`
- Full run: set `STAGES = ""` and use **Save Version -> Save & Run All (Commit)**. If training stops at its time budget, just commit again: it continues where it left off.

In [ ]:
# Cell 1 - get the latest code from GitHub and load secrets. This cell never needs to change.
REPO_URL = "https://github.com/niksalunke/FINETUNINGSLM.git"
SECRETS = ["GITHUB_TOKEN", "HF_TOKEN", "LLM_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY"]

import base64, os, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    _client = UserSecretsClient()
    for _name in SECRETS:
        try:
            os.environ[_name] = _client.get_secret(_name).strip()
            print("secret loaded:", _name)
        except Exception:
            pass
except ImportError:
    print("not running on Kaggle - using environment variables")

_token = os.environ.get("GITHUB_TOKEN", "")
_auth = base64.b64encode(f"x-access-token:{_token}".encode()).decode() if _token else ""
_git = ["git"] + (["-c", f"http.extraHeader=Authorization: Basic {_auth}"] if _token else [])

os.chdir("/kaggle/working")
if os.path.isdir("FINETUNINGSLM/.git"):
    _r = subprocess.run(_git + ["-C", "FINETUNINGSLM", "pull", "--quiet", "--ff-only"], capture_output=True, text=True)
else:
    subprocess.run(["rm", "-rf", "FINETUNINGSLM"])
    _r = subprocess.run(_git + ["clone", "--quiet", REPO_URL, "FINETUNINGSLM"], capture_output=True, text=True)
if _r.returncode != 0:
    _msg = _r.stderr
    for _s in (_token, _auth):
        if _s:
            _msg = _msg.replace(_s, "***")
    raise SystemExit("git failed (check the GITHUB_TOKEN secret is attached and can read the repo):" + chr(10) + _msg)

os.chdir("/kaggle/working/FINETUNINGSLM")
print("code version:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# Cell 2 - run the pipeline. "" = the stages listed in kaggle/config.toml (use this for Save & Run All).
# For a quick interactive check use "setup,data,dryrun".
STAGES = ""

_flag = f"--stages {STAGES}" if STAGES else ""
!python kaggle/run_pipeline.py {_flag}